In [1]:
import pandas as pd
import numpy as np

RAW_DIR = "../data/raw/"

In [ ]:
def load_category_file(filename, label, n_per_rating=400, seed=42):
    """
    Load one category TSV file.
    Sample n_per_rating rows PER STAR RATING to ensure diversity.
    Total rows per file = n_per_rating × 5 ratings = 2000 rows max.
    """
    filepath = RAW_DIR + filename

    print(f"Loading {filename}...")

    # Read in chunks to handle large files efficiently
    chunks = []
    chunk_size = 50000

    for chunk in pd.read_csv(filepath, sep='\t', chunksize=chunk_size,
                              on_bad_lines='skip', low_memory=False):
        chunks.append(chunk)
        # Stop after reading enough data (don't need the whole file)
        if sum(len(c) for c in chunks) > 500000:
            break

    df = pd.concat(chunks, ignore_index=True)
    print(f"  Loaded {len(df)} rows from {filename}")

    # Keep only essential columns
    keep_cols = ['product_title', 'product_category', 'star_rating',
                 'review_headline', 'review_body', 'helpful_votes',
                 'total_votes', 'verified_purchase']

    existing_cols = [c for c in keep_cols if c in df.columns]
    df = df[existing_cols].copy()

    # Drop rows with missing review text or product title
    df = df.dropna(subset=['review_body', 'product_title'])
    df = df[df['review_body'].str.len() > 30]  # Remove very short reviews

    # Clean star_rating — must be 1–5
    df['star_rating'] = pd.to_numeric(df['star_rating'], errors='coerce')
    df = df[df['star_rating'].between(1, 5)]

    # Stratified sample: n_per_rating rows per star level
    sampled = []
    for rating in [1, 2, 3, 4, 5]:
        subset = df[df['star_rating'] == rating]
        n = min(n_per_rating, len(subset))
        sampled.append(subset.sample(n, random_state=seed))

    df_sampled = pd.concat(sampled, ignore_index=True)

    # Add label and source tracking
    df_sampled['cognitive_label'] = label
    df_sampled['source_file'] = filename

    print(f"  Final sample: {len(df_sampled)} rows | Label: {label}")
    return df_sampled


# ── Load the 4 clean System 1 files 
df_beauty   = load_category_file("amazon_reviews_us_Beauty_v1_00.tsv",       label=1)
df_apparel  = load_category_file("amazon_reviews_us_Apparel_v1_00.tsv",      label=1)
df_grocery  = load_category_file("amazon_reviews_us_Grocery_v1_00.tsv",      label=1)
df_pets     = load_category_file("amazon_reviews_us_Pet_Products_v1_00.tsv", label=1)

# ── Load the 1 clean System 2 file 
df_electronics = load_category_file("amazon_reviews_us_Electronics_v1_00.tsv", label=0)

Loading amazon_reviews_us_Beauty_v1_00.tsv...
  Loaded 550000 rows from amazon_reviews_us_Beauty_v1_00.tsv
  Final sample: 2000 rows | Label: 1
Loading amazon_reviews_us_Apparel_v1_00.tsv...
  Loaded 550000 rows from amazon_reviews_us_Apparel_v1_00.tsv
  Final sample: 2000 rows | Label: 1
Loading amazon_reviews_us_Grocery_v1_00.tsv...
  Loaded 550000 rows from amazon_reviews_us_Grocery_v1_00.tsv
  Final sample: 2000 rows | Label: 1
Loading amazon_reviews_us_Pet_Products_v1_00.tsv...
  Loaded 550000 rows from amazon_reviews_us_Pet_Products_v1_00.tsv
  Final sample: 2000 rows | Label: 1
Loading amazon_reviews_us_Electronics_v1_00.tsv...
  Loaded 550000 rows from amazon_reviews_us_Electronics_v1_00.tsv
  Final sample: 2000 rows | Label: 0


In [ ]:
def load_mixed_category(filename, system1_keywords, system2_keywords,
                        n_per_class=500, seed=42):
    """
    Load a mixed category and split rows into S1/S2 based on product_title keywords.
    Rows that don't match any keyword are discarded as ambiguous.
    """
    filepath = RAW_DIR + filename
    print(f"Loading mixed category: {filename}...")

    chunks = []
    for chunk in pd.read_csv(filepath, sep='\t', chunksize=50000,
                              on_bad_lines='skip', low_memory=False):
        chunks.append(chunk)
        if sum(len(c) for c in chunks) > 300000:
            break

    df = pd.concat(chunks, ignore_index=True)
    df = df.dropna(subset=['review_body', 'product_title'])
    df = df[df['review_body'].str.len() > 30]
    df['star_rating'] = pd.to_numeric(df['star_rating'], errors='coerce')
    df = df[df['star_rating'].between(1, 5)]

    title_lower = df['product_title'].str.lower()

    # Apply keyword rules
    mask_s1 = title_lower.str.contains('|'.join(system1_keywords), na=False)
    mask_s2 = title_lower.str.contains('|'.join(system2_keywords), na=False)

    # Exclude rows matching BOTH (genuinely ambiguous)
    df_s1 = df[mask_s1 & ~mask_s2].copy()
    df_s2 = df[mask_s2 & ~mask_s1].copy()

    df_s1['cognitive_label'] = 1
    df_s2['cognitive_label'] = 0
    df_s1['source_file'] = filename + "_S1"
    df_s2['source_file'] = filename + "_S2"

    # Sample equal amounts
    df_s1 = df_s1.sample(min(n_per_class, len(df_s1)), random_state=seed)
    df_s2 = df_s2.sample(min(n_per_class, len(df_s2)), random_state=seed)

    print(f"  S1 rows: {len(df_s1)} | S2 rows: {len(df_s2)}")
    return df_s1, df_s2


# ── Baby category keywords 
BABY_S1_KEYWORDS = ['toy', 'plush', 'stuffed', 'doll', 'gift', 'costume',
                    'blanket', 'book', 'rattle', 'teether', 'bath']
BABY_S2_KEYWORDS = ['monitor', 'thermometer', 'car seat', 'safety', 'gate',
                    'sterilizer', 'nasal', 'scale', 'camera', 'humidifier']

baby_s1, baby_s2 = load_mixed_category(
    "amazon_reviews_us_Baby_v1_00.tsv",
    BABY_S1_KEYWORDS, BABY_S2_KEYWORDS, n_per_class=400
)

# ── Sports category keywords
SPORTS_S1_KEYWORDS = ['yoga', 'water bottle', 'resistance band', 'foam roller',
                      'protein', 'supplement', 'leggings', 'tank', 'shorts',
                      'headband', 'jump rope', 'mat']
SPORTS_S2_KEYWORDS = ['bicycle', 'treadmill', 'gps watch', 'kayak', 'rowing',
                      'elliptical', 'weight bench', 'climbing', 'scope',
                      'golf club', 'helmet', 'cycling computer']

sports_s1, sports_s2 = load_mixed_category(
    "amazon_reviews_us_Sports_v1_00.tsv",
    SPORTS_S1_KEYWORDS, SPORTS_S2_KEYWORDS, n_per_class=400
)

Loading mixed category: amazon_reviews_us_Baby_v1_00.tsv...
  S1 rows: 400 | S2 rows: 400
Loading mixed category: amazon_reviews_us_Sports_v1_00.tsv...
  S1 rows: 400 | S2 rows: 400


In [4]:
# Combine all sources
all_frames = [
    df_beauty, df_apparel, df_grocery, df_pets,  # Pure S1
    df_electronics,                               # Pure S2
    baby_s1, sports_s1,                          # Mixed → S1
    baby_s2, sports_s2                           # Mixed → S2
]

df_merged = pd.concat(all_frames, ignore_index=True)

# Standardize column names
df_merged = df_merged.rename(columns={
    'product_title':    'product_name',
    'review_body':      'review_text',
    'review_headline':  'review_title',
    'product_category': 'category'
})

# Quick balance check
print("\n=== MERGE SUMMARY ===")
print(f"Total rows:   {len(df_merged)}")
print(f"System 1:     {(df_merged.cognitive_label == 1).sum()}")
print(f"System 2:     {(df_merged.cognitive_label == 0).sum()}")
print(f"\nSource breakdown:")
print(df_merged['source_file'].value_counts())

# Save checkpoint
df_merged.to_csv("../data/processed/merged_raw.csv", index=False)
print("\nSaved → ../data/processed/merged_raw.csv")


=== MERGE SUMMARY ===
Total rows:   11600
System 1:     8800
System 2:     2800

Source breakdown:
source_file
amazon_reviews_us_Beauty_v1_00.tsv          2000
amazon_reviews_us_Apparel_v1_00.tsv         2000
amazon_reviews_us_Grocery_v1_00.tsv         2000
amazon_reviews_us_Pet_Products_v1_00.tsv    2000
amazon_reviews_us_Electronics_v1_00.tsv     2000
amazon_reviews_us_Baby_v1_00.tsv_S1          400
amazon_reviews_us_Sports_v1_00.tsv_S1        400
amazon_reviews_us_Baby_v1_00.tsv_S2          400
amazon_reviews_us_Sports_v1_00.tsv_S2        400
Name: count, dtype: int64

Saved → ../data/processed/merged_raw.csv
